# Emergence of Thinking: Model Evaluation Setup & Execution

This notebook automates the environment setup, script patching, and parallel evaluation process for the [Emergence-of-Thinking](https://github.com/GuanghaoYe/Emergence-of-Thinking) repository.

### 📌 Notebook Workflow

*   **Step 1: Environment Initialization:** Clones the target repository and migrates the `math500` dataset into the correct evaluation directory.
*   **Step 2: Custom Script Injection:** Overwrites the default `eval_math_data_parallel.py` and `evaluate.py` scripts with custom logic. This includes enhanced token counting, truncation for absurdly long predictions, and SymPy memory-leak fixes. You need to set your own Hugging Face User Access Token
*   **Step 3: Evaluation Configuration:** Dynamically generates the `eval_config.yaml` file to point exactly to your local model directories and experiment paths.
*   **Step 4: Model Merging & Execution:** Uses `openrlhf` to combine the base Llama model with your SFT LoRA checkpoints, then triggers the data-parallel math evaluation.

---


In [ ]:
import os
import shutil
from pathlib import Path

# Dynamically find your true local repository root
current_dir = Path.cwd()
LOCAL_ROOT = None

# Walk upwards looking for .git or the evaluation folder
for parent in [current_dir] + list(current_dir.parents):
    if (parent / ".git").exists() or (parent / "evaluation").exists():
        LOCAL_ROOT = parent
        break

if LOCAL_ROOT is None:
    # Fallback: assume root is 1 level up from current_dir
    LOCAL_ROOT = current_dir.parents[0]

REPO_NAME = "Emergence-of-Thinking"
repo_path = LOCAL_ROOT / REPO_NAME

# =====================================================================
# FORCE CLEANUP: 
# Uncomment the lines below to wipe the clone folder for a clean slate
# =====================================================================
# if repo_path.exists():
#     print(f"Removing existing directory: {repo_path}")
#     shutil.rmtree(repo_path)

# Clone the repository into the EXACT specified path (Fixed)
if not repo_path.exists():
    print(f"Cloning {REPO_NAME} into {repo_path}...")
    # Pass the Python variable {repo_path} into the bash command 
    !git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git "{repo_path}"
else:
    print(f"'{REPO_NAME}' already exists at {repo_path}. Skipping clone.")

# Force the current working directory to the absolute path of the repo
os.chdir(repo_path)
print(f"Current Working Directory forcibly set to: {os.getcwd()}")

# Copy math500 and aime24 to cloned repo
base_dest_dir = repo_path / "evaluation/data"

# List all the dataset folders you want to copy
folders_to_copy = ["math500", "aime24"]

for folder_name in folders_to_copy:
    src_dir = LOCAL_ROOT / f"evaluation/{folder_name}"
    
    dest_dir = base_dest_dir / folder_name

    if src_dir.exists():
        shutil.copytree(src_dir, dest_dir, dirs_exist_ok=True)
        print(f"Successfully copied {folder_name} to {dest_dir}/")
    else:
        print(f"Warning: Source dir not found at '{src_dir}'.")

**Overwrite evaluation/eval_math_data_parallel.py**

Set your Hugging Face User Access Token

Comment if re-initialization is not needed

In [ ]:
# =====================================================================
# COMMENT THE LINES BELOW IF YOU DO NOT NEED TO RE-INITIALIZE THE REPOSITORY
# =====================================================================
%%writefile evaluation/eval_math_data_parallel.py
"""
Evaluate an LLM on a dataset. Use data parallelism (multiple model replica)
Therefore, if 8 gpus are used, then 8 models are created and each model processes 1/8 of the dataset.
Does not support tensor paralelism due to a bug.
TODO: support both data paralellism and model parallelism.
"""

import os
import random
import argparse
import time
import json
import logging
import multiprocessing
import traceback
import pathlib
import functools
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN" # set HF_TOKEN to avoid some potential issues with loading models from HuggingFace Hub. You can replace it with your own token if needed.
import torch
# from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from tqdm import tqdm
import numpy as np

# from vllm.lora.request import LoRARequest

from omegaconf import OmegaConf

from evaluation import data_loader
from evaluation import parser
from evaluation import model_utils
from evaluation import python_executor
from evaluation import trajectory
from evaluation import utils
from evaluation import evaluate

logger = logging.getLogger("eval_math_data_parallel.py")
logger.setLevel(logging.INFO)
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_names", default="gsm8k,math", type=str)
    parser.add_argument("--data_dir", default="./data", type=str)
    parser.add_argument("--model_name_or_path", default="gpt-4", type=str)
    parser.add_argument("--output_dir", default="./output", type=str)
    parser.add_argument("--prompt_type", default="tool-integrated", type=str)
    parser.add_argument("--split", default="test", type=str)
    parser.add_argument("--num_test_sample", default=-1, type=int)  # -1 for full data
    parser.add_argument("--seed", default=0, type=int)
    parser.add_argument("--start", default=0, type=int)
    parser.add_argument("--end", default=-1, type=int)
    parser.add_argument("--temperature", default=0, type=float)
    parser.add_argument("--n_sampling", default=1, type=int)
    parser.add_argument("--top_p", default=1, type=float)
    parser.add_argument("--max_tokens_per_call", default=2048, type=int)
    parser.add_argument("--shuffle", action="store_true")
    parser.add_argument("--use_vllm", action="store_true")
    parser.add_argument("--save_outputs", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--use_safetensors", action="store_true")
    parser.add_argument("--num_shots", type=int, default=0)
    parser.add_argument(
        "--apply_chat_template",
        action="store_true",
        help="Apply chat template to prompt.",
    )
    parser.add_argument(
        "--adapt_few_shot",
        action="store_true",
        help="Few shot for multiple-choice questions, zero shot for others.",
    )
    parser.add_argument("--async_evaluate", default=False, action="store_true")
    parser.add_argument("--message_path", default=None, type=str)
    parser.add_argument("--config", default=None, type=str)

    args = parser.parse_args()

    if args.config is not None:
        # Load the YAML configuration file
        yaml_config = OmegaConf.load(args.config)

        # Merge argparse arguments with the YAML configuration file.
        # NOTE: config file takes precedence, there other arguments provided in command line will be ignored.
        args = OmegaConf.merge(OmegaConf.create(vars(args)), yaml_config)

    args.top_p = (
        1 if args.temperature == 0 else args.top_p
    )  # top_p must be 1 when using greedy sampling (vllm)

    # get the number of available GPUs
    if "CUDA_VISIBLE_DEVICES" in os.environ:
        args.data_parallel_size = len(os.environ["CUDA_VISIBLE_DEVICES"].split(","))
    else:
        args.data_parallel_size = torch.cuda.device_count()

    return args

def is_choice(answer):
    """
    whether answer is multiple-choice
    """
    return answer in ["A", "B", "C", "D", "E"]

def is_multi_choice(answer):
    """
    whether answer is multiple-choice and check-all-that-apply
    """
    for c in answer:
        if c not in ["A", "B", "C", "D", "E"]:
            return False
    return True


def prepare_data(data_name, args, parallel_index=None):
    """
    1. load data
    # 2. sample `num_test_sample` from dataset
    3. shuffle
    4. select start and end
    5. get out_file name
    6. if there are processed samples, load them and deduplicate
    """
    examples = data_loader.load_data(data_name, args.split, args.data_dir)

    # sample `num_test_sample` from dataset
    if args.num_test_sample > 0:
        # examples = random.sample(examples, min(args.num_test_sample, len(examples)))
        examples = examples[: args.num_test_sample]

    # shuffle
    if args.shuffle:
        # random.seed(datetime.now().timestamp())
        random.seed(args.seed) # fix the seed across different processes
        random.shuffle(examples)

    # select start and end
    if parallel_index is not None:
        examples = examples[
            parallel_index * len(examples) // args.data_parallel_size : (parallel_index + 1)
            * len(examples)
            // args.data_parallel_size
        ]
    else:
        examples = examples[args.start : len(examples) if args.end == -1 else args.end]

    return examples

def setup(args):
    """
    set up data parallel gpus   
    iterate dataset in data_list
    """
    # load model

    # infer & eval
    data_list = args.data_names.split(",")
    results = []
    for data_name in data_list:

        # NOTE: get out_file name
        # dt_string = datetime.now().strftime("%m-%d_%H-%M")
        model_name = "/".join(args.model_name_or_path.split("/")[-2:]) # take last 2 parts of the path
        out_file_prefix = f"{args.split}_{args.prompt_type}_{args.num_test_sample}_seed{args.seed}_t{args.temperature}"
        output_dir = args.output_dir
        # if not os.path.exists(output_dir):
        #     output_dir = f"outputs/{output_dir}"
        out_file = f"{output_dir}/{data_name}/{model_name}/{out_file_prefix}_s{args.start}_e{args.end}.jsonl"
        os.makedirs(f"{output_dir}/{data_name}", exist_ok=True)

        args.out_file = out_file

        # NOTE generate
        if args.data_parallel_size == 1:
            all_samples, time_use, total_tokens, total_times_max_tokens_generated = generate(data_name, args, 0)
            time_use = [time_use]
            all_tokens = [total_tokens]
            all_times_max_tokens_generated = [total_times_max_tokens_generated]

        else:
            try:
                pool = multiprocessing.Pool(processes=args.data_parallel_size)
                generate_results = pool.starmap(generate, [(data_name, args, parallel_index) for parallel_index in range(args.data_parallel_size)])

                # Concatenate the results
                all_samples = [item for split in generate_results for item in split[0]]
                time_use = [split[1] for split in generate_results]
                all_tokens = [split[2] for split in generate_results]
                all_times_max_tokens_generated = [split[3] for split in generate_results]

            except Exception as e:
                print(f"An error occurred: {e}")
                traceback.print_exc()
            finally:
                pool.close()
                pool.join()


        # sort and deduplicate
        all_samples = sorted(all_samples, key=lambda x: x["idx"])
        all_samples = [all_samples[i] for i in range(len(all_samples)) if i == 0 or all_samples[i]["idx"] != all_samples[i - 1]["idx"]]

        logger.info(len(all_samples))

        # evaluate
        all_samples, result_json = evaluate.evaluate(
            samples=all_samples,
            data_name=data_name,
        )

        utils.save_jsonl(all_samples, out_file)

        average_time_use = np.mean(time_use)
        result_json["average_time_use_in_second"] = average_time_use
        result_json["averate_time_use_in_minite"] = (
            f"{int(average_time_use // 60)}:{int(average_time_use % 60):02d}"
        )
        
        # --- OUR CODE: Count tokens and add to metrics ---
        # --- START ---
        try:
            print(f"Aggregating token counts for {data_name}...")
            total_toks = sum(all_tokens)
            total_times_max_toks_generated = sum(all_times_max_tokens_generated)
            result_json["total_tokens_generated"] = total_toks
            result_json["average_tokens_per_problem"] = total_toks / len(all_samples) if len(all_samples) > 0 else 0
            result_json["total_max_tokens_generated"] = total_times_max_toks_generated
        except Exception as e:
            print(f"Warning: Could not aggregate tokens automatically. Error: {e}")
        # --- END ---

        with open(
            out_file.replace(".jsonl", f"_metrics.json"), "w"
        ) as f:
            json.dump(result_json, f, indent=4)

        results.append(result_json)

    # add "avg" result to data_list and results
    data_list.append("avg")
    results.append(
        {
            "maj_acc": sum([result["maj_acc"] for result in results]) / len(results),
        }
    )

    # print all results
    pad = max([len(data_name) for data_name in data_list])
    print("\t".join(data_name.ljust(pad, " ") for data_name in data_list))
    print("\t".join([f"{result['maj_acc']:.3f}".ljust(pad, " ") for result in results]))

def prepare_model(args):
    """
    load model and tokenizer either vllm or huggingface
    """
    if args.use_vllm:
        tokenizer = None
        if args.apply_chat_template:
            tokenizer = AutoTokenizer.from_pretrained(
                args.model_name_or_path, trust_remote_code=True
            )
        llm = model_utils.load_vllm_with_lora(
            model_name_or_path=args.model_name_or_path,
            gpus=[0],
            seed=args.seed,
            # dtype='bfloat16',
        )

    else:
        llm, tokenizer = model_utils.load_hf_lm_and_tokenizer(
            model_name_or_path=args.model_name_or_path,
            load_in_half=True,
            use_fast_tokenizer=True,
            # use_safetensors=args.use_safetensors,
            use_safetensors=True,
        )
    
    return llm, tokenizer

def generate(data_name, args, parallel_index=None):
    """
    1. prepare data
    2. prepare executor
    4. parse question and answer, add fields, repeat prompts, apply template, stop words
    5. generate (multiple epochs if tora type prompt)
    6. if needed, execute code, sort into remain_prompts and end_prompts.
    """
    # set the gpu index to use for this process
    os.environ["CUDA_VISIBLE_DEVICES"] = str(parallel_index)

    llm, tokenizer = prepare_model(args)
    examples = prepare_data(data_name, args, parallel_index)

    print("=" * 50)
    print("data:", data_name, " ,remain samples:", len(examples))
    if len(examples) > 0:
        print(examples[0])

    # init python executor
    if "pal" in args.prompt_type:
        executor = python_executor.PythonExecutor(get_answer_expr="solution()")
    else:
        executor = python_executor.PythonExecutor(get_answer_from_stdout=True)

    samples = []
    for example in tqdm(examples, total=len(examples)):
        idx = example["idx"]

        # parse question and answer
        example["question"] = parser.parse_question(example, data_name)
        if example["question"] == "":
            continue
        gt_cot, gt_ans = parser.parse_ground_truth(example, data_name)
        example["gt_ans"] = gt_ans
        full_prompt = parser.construct_prompt(example, data_name, args)

        if idx == args.start:
            print(full_prompt)

        sample = {
            "idx": idx,
            "question": example["question"],
            "gt_cot": gt_cot,
            "gt": gt_ans,
            "prompt": full_prompt,
        }

        # add remain fields
        for key in [
            "level",
            "type",
            "unit",
            "solution_type",
            "choices",
            "solution",
            "ques_type",
            "ans_type",
            "answer_type",
            "dataset",
            "subfield",
            "filed",
            "theorem",
            "answer",
        ]:
            if key in example:
                sample[key] = example[key]
        samples.append(sample)

    # repeat n times
    input_prompts = [
        sample["prompt"] for sample in samples for _ in range(args.n_sampling)
    ]
    # apply template
    if args.apply_chat_template:
        input_prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt.strip()}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for prompt in input_prompts
        ]
    remain_prompts = input_prompts
    remain_prompts = [(i, prompt) for i, prompt in enumerate(remain_prompts)]
    end_prompts = []

    # support tora type prompt.
    max_func_call = 1 if args.prompt_type in ["cot", "pal"] else 4

    stop_words = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<|end_of_text|>", "<|eom_id|>"]

    # add stop words
    if args.prompt_type in ["cot"]:
        stop_words.append("\n\nQuestion:")
    if args.prompt_type in ["pal", "tool-integrated", "jiuzhang_tora"]:
        stop_words.extend(["\n\n---", "```output"])
    elif args.prompt_type in ["wizard_zs", "platypus_fs"]:
        stop_words.extend(["Instruction", "Response"])
    elif "jiuzhang" in args.prompt_type:
        stop_words.append("\n\n## Question")
    elif "numina" in args.prompt_type:
        stop_words.append("\n### Problem")
    elif "pure" in args.prompt_type:
        stop_words.append("\n\n\n")

    # start inference
    # measure time use
    start_time = time.time()
    for epoch in range(max_func_call):
        print("-" * 20, "Epoch", epoch)
        current_prompts = remain_prompts
        if len(current_prompts) == 0:
            break

        # get all outputs
        prompts = [item[1] for item in current_prompts]
        if args.use_vllm:
            outputs = llm( # LLM.generate
                prompts,
                SamplingParams(
                    temperature=args.temperature,
                    top_p=args.top_p,
                    max_tokens=args.max_tokens_per_call,
                    n=1,
                    stop=stop_words,
                    stop_token_ids=(
                        [151645, 151643]
                        if "qwen2" in args.model_name_or_path.lower()
                        else None
                    ),
                ),
            )

            outputs = sorted(
                outputs, key=lambda x: int(x.request_id)
            )  # sort outputs by request_id
            outputs = [output.outputs[0].text for output in outputs]
        else:
            outputs = model_utils.generate_completions(
                model=llm,
                tokenizer=tokenizer,
                prompts=prompts,
                max_new_tokens=args.max_tokens_per_call,
                batch_size=16,
                stop_id_sequences=stop_words,
                temperature=args.temperature,
                top_p=args.top_p,
                do_sample=(args.temperature > 0),
            )

        assert len(outputs) == len(current_prompts)

        # process all outputs
        remain_prompts = []
        remain_codes = []
        for (i, query), output in zip(current_prompts, outputs):
            output = output.rstrip()
            query += output
            if args.prompt_type == "pal":
                remain_prompts.append((i, query))
                if "```python" in output:
                    output = trajectory.extract_program(query)
                remain_codes.append(output)
            elif args.prompt_type == "cot":
                end_prompts.append((i, query))
            elif "boxed" not in output and output.endswith("```"):
                program = trajectory.extract_program(query)
                remain_prompts.append((i, query))
                remain_codes.append(program)
            else:
                end_prompts.append((i, query))

        # execute the remain prompts
        remain_results = executor.batch_apply(remain_codes)
        for k in range(len(remain_prompts)):
            i, query = remain_prompts[k]
            res, report = remain_results[k]
            exec_result = res if res else report
            if "pal" in args.prompt_type:
                exec_result = "\\boxed{" + exec_result + "}"
            exec_result = f"\n```output\n{exec_result}\n```\n"
            query += exec_result
            # not end
            if epoch == max_func_call - 1:
                query += "\nReach max function call limit."
            remain_prompts[k] = (i, query)

    # unsolved samples
    print("Unsolved samples:", len(remain_prompts))
    end_prompts.extend(remain_prompts)
    # sort by idx
    end_prompts = sorted(end_prompts, key=lambda x: x[0])

    # remove input_prompt from end_prompt
    codes = []
    assert len(input_prompts) == len(end_prompts)
    for i in range(len(input_prompts)):
        _, end_prompt = end_prompts[i]
        code = end_prompt.split(input_prompts[i])[-1].strip()
        for stop_word in stop_words:
            if stop_word in code:
                code = code.split(stop_word)[0].strip()
        codes.append(code)

    # extract preds
    results = [
        parser.run_execute(executor, code, args.prompt_type, data_name) for code in codes
    ]
    time_use = time.time() - start_time

    # put results back to examples
    all_samples = []
    for i, sample in enumerate(samples):
        code = codes[i * args.n_sampling : (i + 1) * args.n_sampling]
        result = results[i * args.n_sampling : (i + 1) * args.n_sampling]
        preds = [item[0] for item in result]
        reports = [item[1] for item in result]
        for j in range(len(preds)):
            if is_choice(sample["gt"]) and not is_choice(preds[j]):
                preds[j] = parser.choice_answer_clean(code[j])
            elif is_multi_choice(sample["gt"]) and not is_multi_choice(preds[j]):
                # remove any non-choice char
                preds[j] = "".join(
                    [c for c in preds[j] if c in ["A", "B", "C", "D", "E"]]
                )

        sample.pop("prompt")
        sample.update({"code": code, "pred": preds, "report": reports})
        all_samples.append(sample)
        
    # --- Token Counting Optimization: Done within the child process ---
    total_tokens_in_process = 0
    total_times_max_tokens_generated_in_process = 0
    try:
        tok_for_counting = tokenizer if tokenizer is not None else AutoTokenizer.from_pretrained(args.model_name_or_path, trust_remote_code=True)
        for s in all_samples:
            for gen_text in s.get("code", []):
                safe_text = str(gen_text)[:20000]
                gen_token = len(tok_for_counting.encode(safe_text))
                if gen_token >= args.max_tokens_per_call:
                    total_times_max_tokens_generated_in_process += 1
                total_tokens_in_process += gen_token
                
    except Exception as e:
        print(f"Warning: Could not count tokens automatically. Error: {e}")
    # --- END ---
    for sample in all_samples:
        if "code" in sample:
            # Truncate the raw outputs to 5,000 characters maximum
            sample["code"] = [str(c)[:30000] if c else "" for c in sample["code"]]
        if "pred" in sample:
            # Truncate predictions to 1,000 characters
            sample["pred"] = [str(p)[:10000] if p else "" for p in sample["pred"]]
        if "report" in sample:
            sample["report"] = [str(r)[:10000] if r else "" for r in sample["report"]]
    
    return all_samples, time_use, total_tokens_in_process, total_times_max_tokens_generated_in_process

if __name__ == "__main__":
    args = parse_args()
    utils.set_seed(args.seed)
    setup(args)

**Overwrite evaluation/evaluate.py**

Cmment if re-initialization is not needed

In [ ]:
# =====================================================================
# COMMENT THE LINES BELOW IF YOU DO NOT NEED TO RE-INITIALIZE THE REPOSITORY
# =====================================================================
%%writefile evaluation/evaluate.py
import argparse

import numpy as np
from tqdm import tqdm

from evaluation import grader
from evaluation import parser
from evaluation import utils

from collections import Counter, defaultdict

def majority_voting(samples):
    acc = 0
    for sample in samples:
        candidate_answers = Counter()
        candidate_answer_correctness = defaultdict(list)
        for pred, score in zip(sample['pred'], sample['score']):
            candidate_answers[pred] += 1
            candidate_answer_correctness[pred].append(score)
                
        # pick the most common answer
        most_common_answer = candidate_answers.most_common(1)[0][0]
        acc += Counter(candidate_answer_correctness[most_common_answer]).most_common(1)[0][0]
    
    pass_rate = acc / len(samples)

    return pass_rate


def evaluate(data_name, samples: list=None, file_path: str=None, max_num_samples=None):
    assert samples or file_path, "samples or file_path must be provided"
    if not samples:
        samples = list(utils.load_jsonl(file_path))
    if 'idx' in samples[0]:
        samples = {sample['idx']: sample for sample in samples}.values()
        samples = sorted(samples, key=lambda x: x['idx']) 
    else:
        samples = [dict(idx=idx, **sample) for idx, sample in enumerate(samples)]

    if max_num_samples:
        print(f"max_num_samples: {max_num_samples} / {len(samples)}")
        samples = samples[:max_num_samples]
    
    from collections import defaultdict
    level_correct = defaultdict(int)
    level_total = defaultdict(int)

    i=0
    for sample in samples:
        i+=1
        sample['gt_cot'], sample['gt'] = parser.parse_ground_truth(sample, data_name)
        
        # 1. Truncate absurdly long predictions before they hit the grader
        safe_preds = []
        for pred in sample['pred']:
            if pred and isinstance(pred, str):
                safe_preds.append(pred[:20000]) 
            else:
                safe_preds.append(pred)
                
        # 2. Run the grader
        sample['score'] = [grader.math_equal(pred, sample['gt']) for pred in safe_preds]
        
        # 3. Clear SymPy's memory leak cache after every question
        try:
            import sympy
            sympy.core.cache.clear_cache()
        except Exception:
            pass 

    result_json = {
        "num_samples": len(samples),
        "empty_samples": len([s for s in samples if not s['pred'][-1]]),
        "average_acc": np.mean([np.mean(sample['score']) for sample in samples]),
        "pass_acc": np.mean([np.any(sample['score']) for sample in samples]),
        "maj_acc": majority_voting(samples)
    }

    # each type score
    if "type" in samples[0]:
        type_scores = {}
        for sample in samples:
            if sample['type'] not in type_scores:
                type_scores[sample['type']] = []
            type_scores[sample['type']].append(sample['score'][-1])
        type_scores = {k: np.round(np.array(v).mean() * 100, decimals=1) for k, v in type_scores.items()}
        type_scores = {k: v for k, v in sorted(type_scores.items(), key=lambda item: item[0])}
        result_json['type_acc'] = type_scores

    print(result_json)
    return samples, result_json


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_name", type=str, default="math")
    parser.add_argument("--prompt_type", type=str, default="tool-integrated")
    parser.add_argument("--file_path", type=str, default=None, required=True)
    parser.add_argument("--max_num_samples", type=int, default=None)src_file
    args = parser.parse_args()
    return args

if __name__ == "__main__":
    args = parse_args()
    evaluate(data_name=args.data_name, file_path=args.file_path,
             max_num_samples=args.max_num_samples)

**Configuration of Evaluation**

In [ ]:
%%writefile eval_config.yaml
model_name_or_path: "experiments/recreation/Llama1B_gradientaccumulation_dynamica/merged"
data_names: math500 # for math500 dataset, else define the target dataset
data_dir: evaluation/data
split: test
# uncomment the following line for majority voting evaluation (token efficiency test). Change the n_samplings to define the number of samples per question
# n_sampling: 3
# temperature: 0.7
# top_p: 0.95
# output_dir: "eval_results_final/recreation/token_efficiency_test" # for majority voting evaluation, uncomment this line and comment the next one if needed
output_dir: "eval_results_final/expansions" 
num_workers: 1
num_test_sample: -1 # -1 for the whole dataset, else define the number of samples
max_tokens_per_call: 1500
batch_size: 1
use_vllm: false
prompt_type: llama
apply_chat_template: true


In [ ]:
# 1. Combine LoRA adapters and output the merged weights
python -m openrlhf.cli.lora_combiner \
    --model_path path_to_base_model \
    --lora_path experiments/recreation/path_to_lora_adaptors_of_trained_model \
    --output_path experiments/recreation/experiment_25/path_to_merged_model

# 2. Run the evaluation script utilizing your config setup
python -m evaluation.eval_math_data_parallel --config ./eval_config.yaml